# 🧠 GraphRAG — Build FAISS Index on Colab

This notebook builds the FAISS vector index from ~600 Wikipedia articles
and saves `faiss_index.bin` + `faiss_chunks.pkl` for use in the project.

**Run this once on Colab (free GPU) then download the files into `data/`.**

Steps:
1. Install dependencies
2. Clone the repo (or mount Drive)
3. Load Wikipedia dataset via HuggingFace
4. Build FAISS index
5. Download `faiss_index.bin` to include in repo

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q groq sentence-transformers faiss-cpu evaluate bert-score \
    transformers pyTigerGraph python-dotenv huggingface-hub pandas numpy wikipedia datasets

In [ ]:
# ── Step 2: Clone the repo ─────────────────────────────────────────────────────
import os, sys

REPO_URL = 'https://github.com/YOUR_USERNAME/graph-RAG.git'  # ← update this!
REPO_DIR = '/content/graph-RAG/graphrag_project'

if not os.path.exists('/content/graph-RAG'):
    os.system(f'git clone {REPO_URL} /content/graph-RAG')

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ── Step 3: Set API keys (Groq required to build full pipeline) ───────────────
import os
os.environ['GROQ_API_KEY'] = 'gsk_YOUR_KEY_HERE'  # ← required
os.environ['HF_TOKEN']     = 'hf_YOUR_TOKEN_HERE'  # ← optional but recommended
print('Keys set.')

In [ ]:
# ── Step 4: Load Wikipedia via HuggingFace datasets (~600 articles) ───────────
from datasets import load_dataset
import re, pickle, os

print('Loading Wikipedia dataset from HuggingFace...')
ds = load_dataset(
    'wikimedia/wikipedia',
    '20220301.en',
    split='train[:600]',
    trust_remote_code=True
)
print(f'Loaded {len(ds)} articles from HuggingFace Wikipedia')

def chunk_text(text, max_tokens=256):
    """Split text into ~256-token chunks (1 token ≈ 4 chars)."""
    max_chars = max_tokens * 4
    words = text.split()
    chunks, current, current_len = [], [], 0
    for word in words:
        wl = len(word) + 1
        if current_len + wl > max_chars and current:
            chunks.append(' '.join(current))
            current, current_len = [word], wl
        else:
            current.append(word)
            current_len += wl
    if current:
        chunks.append(' '.join(current))
    return [c for c in chunks if len(c.strip()) > 60]

all_chunks = []
for i, row in enumerate(ds):
    text = row.get('text', '') or ''
    if len(text) < 100:
        continue
    chunks = chunk_text(text, max_tokens=256)
    all_chunks.extend(chunks[:20])  # max 20 chunks per article
    if (i + 1) % 100 == 0:
        print(f'  [{i+1}/600] {len(all_chunks)} chunks so far...')

total_tokens_est = len(all_chunks) * 256
print(f'\n✅ Wikipedia: {len(ds)} articles → {len(all_chunks)} chunks (~{total_tokens_est:,} token-slots)')
print(f'   Requirement: ≥600 articles, ~1.5M tokens')
print(f'   Actual: {len(ds)} articles, ~{total_tokens_est:,} tokens')

# Save wiki chunks cache
os.makedirs('data', exist_ok=True)
with open('data/wiki_chunks.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)
print('Saved data/wiki_chunks.pkl')

In [ ]:
# ── Step 5: Build FAISS index ─────────────────────────────────────────────────
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle

print('Loading sentence-transformers model...')
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f'Encoding {len(all_chunks)} chunks (this takes ~5-10 min on CPU, <2 min on GPU)...')
embeddings = model.encode(all_chunks, convert_to_numpy=True, show_progress_bar=True).astype('float32')

print('Building FAISS IndexFlatL2...')
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# Save FAISS index
faiss.write_index(index, 'data/faiss_index.bin')
with open('data/faiss_chunks.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)

print(f'\n✅ FAISS index built: {index.ntotal} vectors, {len(all_chunks)} chunks')
print('Saved: data/faiss_index.bin + data/faiss_chunks.pkl')

In [ ]:
# ── Step 6: Verify the index works ────────────────────────────────────────────
test_query = 'What is machine learning?'
qv = model.encode([test_query], convert_to_numpy=True).astype('float32')
_, idx = index.search(qv, 3)
print(f'Top-3 retrieved chunks for "{test_query}":')
for i, j in enumerate(idx[0]):
    print(f'  [{i+1}] {all_chunks[j][:120]}...')

In [ ]:
# ── Step 7: Download files to local machine ───────────────────────────────────
# Downloads faiss_index.bin + faiss_chunks.pkl to include in your git repo.
# Place them in: graphrag_project/data/
try:
    from google.colab import files
    print('Downloading faiss_index.bin...')
    files.download('data/faiss_index.bin')
    print('Downloading faiss_chunks.pkl...')
    files.download('data/faiss_chunks.pkl')
    print('Downloading wiki_chunks.pkl...')
    files.download('data/wiki_chunks.pkl')
    print('\n✅ Download complete!')
    print('Place these files in your repo at: graphrag_project/data/')
except ImportError:
    print('Not running in Colab. Files are saved at data/faiss_index.bin')
    print(f'Index size: {os.path.getsize("data/faiss_index.bin"):,} bytes')